In [ ]:
import numpy as np
import pandas as pd
import pickle

In [ ]:
data = pd.read_csv('cars.csv')

In [ ]:

# Step 1: Fix formatting issues
# Price (`pu`)
data['pu'] = data['pu'].str.replace(',', '').astype(float)

# Max Power
data['Max Power'] = data['Max Power'].str.extract(r'(\d+\.?\d*)bhp').astype(float)

# Max Torque
data['Max Torque'] = data['Max Torque'].str.extract(r'(\d+\.?\d*)Nm').astype(float)

# Top Speed
data['Top Speed'] = data['Top Speed'].str.extract(r'(\d+\.?\d*)').astype(float)

# Acceleration
data['Acceleration'] = data['Acceleration'].str.extract(r'(\d+\.?\d*)').astype(float)

# Mileage (`mileage_new`)
data['mileage_new'] = data['mileage_new'].str.extract(r'(\d+\.?\d*)').astype(float)

data["car_age"] = 2025 - data["myear"]


In [ ]:
# Turbo Charger and Super Charger
data['Turbo Charger'] = data['Turbo Charger'].str.lower().map({'yes': 1, 'no': 0})
data['Super Charger'] = data['Super Charger'].str.lower().map({'yes': 1, 'no': 0})

# One-hot encode 'carType' and drop the original column
if "ft" in data.columns:
    data = pd.get_dummies(data, columns=["ft"], prefix="ft")


In [ ]:
data["pu"] = np.log1p(data["pu"])  # log(1 + x) for stability

In [ ]:

features = ["pu", "Max Power", "Max Torque", "Top Speed", "Acceleration", "mileage_new", "Turbo Charger",
             "Super Charger", "km_driven", 'car_age' , 'ft_CNG',
            'ft_Diesel', 'ft_Electric', 'ft_LPG', 'ft_Petrol', "Gear Box"]

In [ ]:
X = data[features]
y = data['tt']
# Encode target variable (`tt`)
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(y)


In [ ]:
with open('scaler.pkl', 'rb') as file:
    loaded_scaler = pickle.load(file)

with open('imputer.pkl', 'rb') as file:
    loaded_imputer = pickle.load(file)

with open('smote.pkl', 'rb') as file:
    loaded_smote = pickle.load(file)

In [ ]:

X = loaded_scaler.transform(X)

In [ ]:

X = loaded_imputer.transform(X)

In [ ]:

X = loaded_smote.transform(X)

In [ ]:
from tensorflow import eras 
loaded_model = keras.models.load_model('trained_model.keras')
loaded_model.summary()

In [ ]:
predictions = loaded_model.predict(X)

In [ ]:

# Evaluate
test_loss, test_accuracy = loaded_model.evaluate(X, y, verbose=0)
print(f"Validation Accuracy: {test_accuracy}")


In [ ]:
for actual,pred in zip(y, predictions):
    print(f'Actual: {actual}, pred: {pred}') 